# VibeVoice Pipeline on Colab

Runs the full Video Generation Pipeline on Colab's free GPU (T4).


## 0. Clone repositories


In [ ]:
# ── GPU check ────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), (
    "No GPU! Top menu: Runtime -> Change runtime type -> T4 GPU, then re-run all cells. "
    "On CPU the model load exhausts RAM and Colab kills the process."
)
print("GPU:", torch.cuda.get_device_name(0))

# ── Model choice ─────────────────────────────────────────────
# "0.5B"  — fast streaming model, fixed .pt voices (free T4 OK)
# "1.5B"  — TRUE voice cloning from a reference WAV (free T4 OK, ~7GB)
# "Large" — the 7B-class model, best quality (needs Colab Pro A100/L4, ~20GB VRAM)
VIBEVOICE_MODEL = "1.5B"

import os
os.environ["VIBEVOICE_MODEL"] = VIBEVOICE_MODEL

!rm -rf audio
!git clone https://github.com/saitejatiru/audiowithvideoremotion.git audio
%cd audio
!rm -rf VibeVoice
if VIBEVOICE_MODEL == "0.5B":
    # microsoft repo has the streaming model code + .pt voices
    !git clone https://github.com/microsoft/VibeVoice.git
else:
    # community fork restores the non-streaming TTS code + reference WAV voices
    !git clone https://github.com/vibevoice-community/VibeVoice.git
import sys; sys.path.insert(0, ".")


## 1. Install dependencies
Installs requirements for TTS, Alignment, Storyboarding, Rendering, and Orchestration.


In [ ]:
!pip install -q fastapi "uvicorn[standard]" soundfile requests nemo-text-processing gradio tenacity "pandas<3.0.0" "huggingface-hub>=1.2.0"
!pip install -q -r align/requirements.txt
!pip install -q openai json-repair

# VibeVoice model package — REQUIRED, run_vibevoice imports `vibevoice`
!pip install -q -e VibeVoice

# Node.js + Remotion dependencies for Phase 4 rendering
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!cd video && npm install
!sudo apt-get install -yq gconf-service libasound2 libatk1.0-0 libc6 libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 libgcc1 libgconf-2-4 libgdk-pixbuf2.0-0 libglib2.0-0 libgtk-3-0 libnspr4 libpango-1.0-0 libpangocairo-1.0-0 libstdc++6 libx11-6 libx11-xcb1 libxcb1 libxcomposite1 libxcursor1 libxdamage1 libxext6 libxfixes3 libxi6 libxrandr2 libxrender1 libxss1 libxtst6 ca-certificates fonts-liberation libappindicator1 libnss3 lsb-release xdg-utils wget

## 2. Set API Keys


In [ ]:
import os

# ── LLM: NVIDIA NIM ──────────────────────────────────────────
# Key from build.nvidia.com (starts with "nvapi-"). Paste it below.
# LLM_MODEL must be a model your account can INVOKE on the hosted endpoint —
# llama-3.3-70b is verified working. NOTE: many catalog models (e.g. Kimi K2.6)
# are self-host-only and 404 with "Function not found for account" — if that
# happens, run the model-probe cell to find one that works for your account.
os.environ["LLM_BASE_URL"] = "https://integrate.api.nvidia.com/v1"
os.environ["LLM_API_KEY"]  = "<YOUR_NVAPI_KEY>"
os.environ["LLM_MODEL"]    = "meta/llama-3.3-70b-instruct"
# Alternative — DeepSeek: base https://api.deepseek.com, model "deepseek-chat"

# ── Visuals: Pixabay (free) for illustrations + stock video ──
# Repo is PUBLIC — do NOT commit your key here. Paste it in THIS Colab cell at
# runtime (it stays in the session only), or use Colab Secrets (🔑 panel):
#   from google.colab import userdata
#   os.environ["PIXABAY_API_KEY"] = userdata.get("PIXABAY_API_KEY")
os.environ["PIXABAY_API_KEY"] = "<PASTE_YOUR_PIXABAY_KEY_HERE>"
# os.environ["PIXABAY_VIDEO"] = "1"   # uncomment to use stock VIDEO b-roll
                                       # (motion, but keyword match is looser)

os.environ["GRADIO_SHARE"] = "1"

# ── Smoke test — RUN cell 2, both lines must say OK before you generate ──
if "<" in os.environ["LLM_API_KEY"]:
    print("!! LLM_API_KEY placeholder — storyboard will be PLAIN TEXT. Paste a real key.")
else:
    from openai import OpenAI
    _c = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
    try:
        _r = _c.chat.completions.create(
            model=os.environ["LLM_MODEL"],
            messages=[{"role": "user", "content": "Reply with the word: ok"}],
            max_tokens=10,
        )
        print("LLM OK:", os.environ["LLM_MODEL"], "->", _r.choices[0].message.content)
    except Exception as e:
        print("=" * 70)
        print("LLM BROKEN:", repr(e))
        print("Try the model-probe cell to find a model your account can invoke.")
        try:
            probe = ["meta/llama-3.3-70b-instruct", "deepseek-ai/deepseek-r1",
                     "meta/llama-3.1-8b-instruct"]
            for m in probe:
                try:
                    _c.chat.completions.create(model=m, messages=[{"role":"user","content":"ok"}], max_tokens=5)
                    print("  ✅ this one works — set LLM_MODEL =", m); break
                except Exception:
                    print("  ❌", m)
        except Exception:
            pass
        print("=" * 70)

# Pixabay check
if "<" in os.environ["PIXABAY_API_KEY"]:
    print("!! PIXABAY_API_KEY placeholder — scenes will have NO illustrations/video. Paste your key above.")
else:
    import requests
    try:
        _n = requests.get("https://pixabay.com/api/", params={
            "key": os.environ["PIXABAY_API_KEY"], "q": "human heart",
            "image_type": "illustration", "per_page": 3}, timeout=10
        ).json().get("totalHits", 0)
        print("PIXABAY OK — 'human heart' illustrations available:", _n)
    except Exception as e:
        print("PIXABAY BROKEN:", repr(e))


## 3. Launch the Full Gradio UI
This will expose the complete pipeline (TTS -> Align -> Storyboard -> Render -> Post-Process) through a public Gradio URL.


In [ ]:
%cd /content/audio
!python app.py
